# Suite CON — Predicates

A predicate is a named Basic rule over symbols, each bound to an object of a schema; it applies to a match, which binds
every symbol. Predicates and their sets are built fluently, as schemas are, and are data: written by their schemas'
names, and read back through a store that resolves them.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Terms } from "@mbse/expressions/Framework";
import { Constraints as C, register } from "@mbse/patterns";
import { Errors, JSON as SchemaJSON, Proxies, Schemas as S, Validators as SV } from "@mbse/schemas/Framework";
import { Contact, Phone, assert, book, raises, same } from "./support.js";

const the = E.variable("the");
const IsAnAdult = new C.OfPredicate.Builder()
  .name("IsAnAdult")
  .description("18 or older")
  .symbols({ the: Contact })
  .rule(the.age.ge(18n))
  .create();
const Numbered = new C.OfPredicate.Builder().name("Numbered").symbols({ the: Phone }).rule(the.has("number")).create();

## CON-01 · A predicate is built fluently: a name, a description, symbols and a rule

In [ ]:
{
  assert(IsAnAdult.name === "IsAnAdult" && IsAnAdult.description === "18 or older" && same(IsAnAdult.symbols.keys(), ["the"]));
  assert(IsAnAdult.symbols.get("the") === Contact && IsAnAdult.rule instanceof E.OfOperation.Data); // a writer, resolved
  assert(Terms.same(IsAnAdult.rule, the.age.ge(18n).data));
  assert(Numbered.description === null && IsAnAdult.validate().length === 0);
  const two = new C.OfPredicate.Builder().symbols({ a: Contact }).symbols(new Map([["b", Phone]])).create();
  assert(same(two.symbols.keys(), ["a", "b"]) && two.rule === null); // symbols add up, in order; nothing validates yet
  const renamed = new C.OfPredicate.Builder(IsAnAdult).name("IsGrown").clone();
  assert(renamed.name === "IsGrown" && IsAnAdult.name === "IsAnAdult" && renamed.rule === IsAnAdult.rule);
  assert(new C.OfPredicate.Builder(renamed).description("grown").update() === renamed && renamed.description === "grown");
  assert(C.OfPredicate.resolve(IsAnAdult) === IsAnAdult);
  assert(C.OfPredicate.resolve((b: C.OfPredicate.Builder) => b.name("X").symbols({ x: Contact }).rule(true)).name === "X");
  raises(TypeError, () => C.OfPredicate.resolve("X"), "expected a predicate or a callable taking its builder, got 'X'");
  raises(Errors.ValueError, () => new C.OfPredicate.Builder(IsAnAdult).create(),
    "create() is only valid without a source instance; use clone() or update()");
  raises(Errors.ValueError, () => new C.OfPredicate.Builder().clone(), "clone() is only valid with a source instance");
  raises(Errors.ValueError, () => new C.OfPredicate.Builder().update(), "update() is only valid with a source instance");
  assert(IsAnAdult.schema_name() === "Patterns.Predicate" && IsAnAdult.owner() === null && IsAnAdult.identity() === IsAnAdult.identity());
}

## CON-02 · Predicates are checked when asked

In [ ]:
{
  const Value = new S.OfObject.Builder().name("Value").create();
  const bad = new C.OfPredicate.Builder().name("Bad").symbols({ a: Value, b: new S.OfObject.Builder().ref().create() }).rule(
    E.operation("pow", E.variable("a"), E.variable("n"))).create();
  assert(same(bad.validate(), ["predicate 'Bad': symbol 'a' needs a named reference object schema",
    "predicate 'Bad': symbol 'b' needs a named reference object schema",
    "predicate 'Bad': 'pow' is not a core operation",
    "predicate 'Bad': argument 1: variable 'n' is not bound"]));
  assert(same(new C.OfPredicate.Builder().create().validate(), ["predicate None: a predicate needs a name",
    "predicate None: a predicate needs a rule"])); // symbols are optional
  const twice = new C.OfSet.Builder().predicates(IsAnAdult, Numbered, new C.OfPredicate.Builder(IsAnAdult).clone()).create();
  assert(same(twice.validate(), ["predicate 'IsAnAdult': defined twice"]));
  raises(Errors.ValueError, () => C.check([bad]), "predicate 'Bad': symbol 'a' needs a named reference object schema; predicate 'Bad'");
  const predicates = C.check([IsAnAdult, (b: C.OfPredicate.Builder) => b.name("Named").symbols({ the: Contact }).rule(the.has("name"))]);
  assert(predicates instanceof C.OfSet.Data && C.check(predicates) === predicates);
}

## CON-03 · A set orders predicates

In [ ]:
{
  const predicates = new C.OfSet.Builder().predicates(IsAnAdult).predicates(Numbered).create();
  assert(same(predicates.predicates, [IsAnAdult, Numbered]) && Object.isFrozen(predicates.predicates)); // in order
  assert(new C.OfSet.Builder().create().predicates.length === 0 && predicates.schema_name() === "Patterns.Set");
  assert(predicates.owner() === null && predicates.identity() === predicates.identity()
    && predicates.identity() !== new C.OfSet.Builder().create().identity());
  const grown = new C.OfSet.Builder(predicates).predicates((b) => b.name("Grown").symbols({ the: Contact }).rule(true)).clone();
  assert(same(grown.predicates.map((p) => p.name), ["IsAnAdult", "Numbered", "Grown"]) && predicates.predicates.length === 2);
}

## CON-04 · Predicates are data: written by their symbols' schema names, read back through a store

In [ ]:
{
  const shared = the.age.ge(18n).data;
  const predicates = new C.OfSet.Builder().predicates(
    new C.OfPredicate.Builder(IsAnAdult).rule(shared).clone(),
    (b) => b.name("Owns").symbols({ c: Contact, p: Phone }).rule(E.operation("implies", shared, E.variable("p").has("number"))),
  ).create();
  const store = new C.OfStore(book()); // resolves the symbols' schemas in the address book
  const text = SchemaJSON.ToJSON(store).Reachable(C.OfSet.Schema, predicates);
  assert(text.includes('"symbols": [{"name": "c", "type": {"named": {"name": "Contact"}}}')); // by name: no store needed to write
  assert(text.split('"kind": "operation"').length - 1 === 4); // get, ge, implies and has: the shared rule's two are written once
  assert(SchemaJSON.ToJSON(C.Builders).Reachable(C.OfSet.Schema, predicates) === text); // any store writes them alike
  const copy = SchemaJSON.FromJSON(store).Reachable(C.OfSet.Schema, text) as C.OfSet.Data;
  assert(JSON.stringify(copy.predicates.map((p) => [p.name, p.description])) === JSON.stringify([["IsAnAdult", "18 or older"], ["Owns", null]]));
  const owns = copy.predicates[1] as C.OfPredicate.Data;
  assert(owns.symbols.get("c") === Contact && owns.symbols.get("p") === Phone); // the store's own
  assert((owns.rule as any).arguments[0] === copy.predicates[0]!.rule); // still shared
  assert(Terms.same(owns.rule, predicates.predicates[1]!.rule));
  assert(SV.Validate(store).Reachable(C.OfSet.Schema, predicates).length === 0); // valid data, by mbse-schemas' rules
  raises(Errors.LookupError, () => SchemaJSON.FromJSON(C.Builders).Reachable(C.OfSet.Schema, text),
    "no schema named 'Contact' in the module or the store"); // reading resolves names, so it needs the store
  const Inline = new S.OfObject.Builder().ref().properties((p) => p.name("n").of((t) => t.as_native(BigInt))).create();
  const loose = new C.OfPredicate.Builder().name("Loose").symbols({ x: Inline }).rule(true).create();
  const back = SchemaJSON.FromJSON(C.Builders).Reachable(C.OfPredicate.Schema,
    SchemaJSON.ToJSON(C.Builders).Reachable(C.OfPredicate.Schema, loose)) as C.OfPredicate.Data;
  const x = back.symbols.get("x") as S.OfObject.Data;
  assert(x !== Inline && same(x.properties.keys(), ["n"])); // an unnamed schema, inline
  const built = store.builder("Patterns.Predicate", IsAnAdult);
  built.property("name", (p: any) => p.value((a: any) => a.as_native((n: any) => n.set("Grown"))));
  assert(built.update() === IsAnAdult && IsAnAdult.name === "Grown" && IsAnAdult.symbols.get("the") === Contact);
  IsAnAdult.name = "IsAnAdult";
}

## CON-05 · Decoding refuses what a predicate or a set cannot hold

In [ ]:
{
  const store = new C.OfStore(book());
  const index = (e: any, i: bigint) => e.property("index", (p: any) => p.value((v: any) => v.as_native((n: any) => n.set(i))));
  const ruleEntry = (target: unknown, i = 0n) => (a: any) => a.add((e: any) => {
    if (target !== null) e.link("argument", (l: any) => l.set(target));
    index(e, i);
  });
  const memberEntry = (target: unknown, i: bigint | null = null) => (a: any) => a.add((e: any) => {
    if (target !== null) e.link("predicate", (l: any) => l.set(target));
    if (i !== null) index(e, i);
  });
  raises(Errors.ValueError, () => store.builder("Patterns.Predicate").adjacency("rule", ruleEntry(the.data)).adjacency(
    "rule", ruleEntry(the.data, 1n)).create(), "a predicate has one rule, got 2");
  raises(TypeError, () => store.builder("Patterns.Predicate").adjacency("rule", ruleEntry(Numbered)).create(),
    "a rule must be an expression");
  raises(Errors.ValueError, () => store.builder("Patterns.Predicate").adjacency("rule", ruleEntry(null)).create(),
    "link 'argument' is not set");
  raises(TypeError, () => store.builder("Patterns.Set").adjacency("predicates", memberEntry(the.data, 0n)).create(),
    "a member must be a predicate");
  raises(Errors.ValueError, () => store.builder("Patterns.Set").adjacency("predicates", memberEntry(null, 0n)).create(),
    "link 'predicate' is not set");
  const unordered = store.builder("Patterns.Set").adjacency("predicates", memberEntry(Numbered)).adjacency(
    "predicates", memberEntry(IsAnAdult, 0n)).adjacency("predicates", memberEntry(IsAnAdult)).create() as C.OfSet.Data;
  assert(same(unordered.predicates, [IsAnAdult, Numbered, IsAnAdult])); // entries without an index come last, in their order
  const blank = store.builder("Patterns.Predicate").create() as C.OfPredicate.Data; // what a snapshot leaves out is absent
  assert(blank.name === null && blank.description === null && blank.symbols.size === 0 && blank.rule === null);
  assert(SchemaJSON.ToJSON(C.Builders)(C.OfPredicate.Schema, blank) === '{"root": "s0", "objects": {"s0": {}}}'); // nothing to write
}

## CON-06 · Another store holds predicates as proxies

In [ ]:
{
  const store = register(new Proxies.OfStore());
  for (const name of ["Patterns.Predicate", "Patterns.Set", "Patterns.Members", "Expressions.OfOperation"]) {
    assert(store.names().includes(name), name);
  }
  assert(register(store) === store); // what it holds already is skipped
  const loose = new C.OfPredicate.Builder().name("Any").symbols({ x: new S.OfObject.Builder().ref().create() }).rule(true).create();
  const proxy = SchemaJSON.FromJSON(store).Reachable(C.OfPredicate.Schema,
    SchemaJSON.ToJSON(C.Builders).Reachable(C.OfPredicate.Schema, loose)) as any;
  assert(proxy.schema_name() === "Patterns.Predicate" && Proxies.store_of(proxy) === store && proxy.name === "Any");
}